# Sequence Labelling: Pediction Sentence

- Sequence: Pediction Sentence
- Labels: Prediction Properties

Tasks
1. Update the prompt from zero-shot to few-show
2. How can we measure the output/labels from the LLMs?
    1. Could use multiple LLMs and perform majority vote for the labels
3. Perform on various datasets
    1. Synthetic data (which is below)
    2. Real datasets
        1.  financial_phrasebank
        2. chronicle2050

> `script_experiments/extract_prediction_properties.py` is where we extract the properties

In [1]:
import os
import sys
import pprint

import pandas as pd

from tqdm import tqdm

notebook_dir = os.getcwd()

sys.path.append(os.path.join(notebook_dir, '../'))

from data_processing import DataProcessing
from prediction_properties import PredictionProperties
from text_generation_models import TextGenerationModelFactory
from vector_stores import ChromaVectorStore, VectorStoreDirector

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Load Data

- Request data

In [3]:
base_data_path = os.path.join(notebook_dir, '../data')

In [4]:
combine_data_path = os.path.join(base_data_path, 'extract_prediction_properties/future-chronicle2050_fpb_synthethic-v1/openai/gpt-oss-120b/')
model_results_path = os.path.join(combine_data_path, 'results.csv')
df = DataProcessing.load_from_file(model_results_path, 'csv', sep=',')
df

,Sentence,Raw Response,Model Name,No Property,Source,Target,Date,Outcome,Dataset Source
0,"By January 1st, 2037, Tesla will have been the first company with 1 million vehicles that are capable of SAE Level 4 autonomy on over 90% of public roads in the contiguous United States, with human-level safety or better, and this capability will be usable by the general public commercially.","{0: [""By"", ""will"", ""have"", ""been"", ""and"", ""this"", ""will"", ""be"", ""usable"", ""by"", ""the"", ""general"", ""public"", ""commercially""], 1: [""Tesla""], 2: [""Tesla""], 3: [""January 1st, 2037""], 4: [""first company with 1 million vehicles that are capable of SAE Level 4 autonomy on over 90% of public roads in the contiguous United States, with human-level safety or better, and this capability will be usable by the general public commercially""]}",openai/gpt-oss-120b,"By, will, have, been, and, this, will, be, usable, by, the, general, public, commercially",Tesla,Tesla,"January 1st, 2037","first company with 1 million vehicles that are capable of SAE Level 4 autonomy on over 90% of public roads in the contiguous United States, with human-level safety or better, and this capability will be usable by the general public commercially",future-chronicle2050_fpb_synthethic-v1
1,An annual average temperature anomaly value above the 1850-1899 baseline will be published in the Berkeley Earth Global Temperature series as 2.0C or higher on or before the 2037 value (published in 2038).,"{\n ""0"": [""An"", ""above"", ""the"", ""will"", ""be"", ""published"", ""in"", ""the"", ""as"", ""or"", ""higher"", ""on"", ""or"", ""before"", ""the"", ""value"", ""(published"", ""in""],\n ""1"": [],\n ""2"": [""annual average temperature anomaly value""],\n ""3"": [""2037"", ""2038""],\n ""4"": [""2.0C or higher""]\n}",openai/gpt-oss-120b,"An, above, the, will, be, published, in, the, as, or, higher, on, or, before, the, value, (published, in",NaN,annual average temperature anomaly value,"2037, 2038",2.0C or higher,future-chronicle2050_fpb_synthethic-v1
2,Private Nonfarm business productivity growth will average over 1.8 percent per year from the first quarter (Q1) of 2020 to the last quarter of 2029 (Q4).,"{0: [""will"", ""from"", ""to""], 1: [], 2: [""Private Nonfarm business productivity growth""], 3: [""the first quarter (Q1) of 2020"", ""the last quarter of 2029 (Q4)""], 4: [""average over 1.8 percent per year""]}",openai/gpt-oss-120b,"will, from, to",NaN,Private Nonfarm business productivity growth,"the first quarter (Q1) of 2020, the last quarter of 2029 (Q4)",average over 1.8 percent per year,future-chronicle2050_fpb_synthethic-v1
3,"No Republican will be President of the USA before December 30, 2039.","{0: [""will"", ""be"", ""before""], 1: [], 2: [""Republican""], 3: [""December 30, 2039""], 4: [""No"", ""President of the USA""]}",openai/gpt-oss-120b,"will, be, before",NaN,Republican,"December 30, 2039","No, President of the USA",future-chronicle2050_fpb_synthethic-v1
4,The market capitalization of Berkshire Hathaway will be greater than Tesla on Warren Buffett's 100th birthday.,"{0: [""The"", ""of"", ""will"", ""be"", ""on""], 1: [], 2: [""Berkshire Hathaway"", ""Tesla""], 3: [""Warren Buffett's 100th birthday""], 4: [""market capitalization"", ""greater than""]}",openai/gpt-oss-120b,"The, of, will, be, on",NaN,"Berkshire Hathaway, Tesla",Warren Buffett's 100th birthday,"market capitalization, greater than",future-chronicle2050_fpb_synthethic-v1
...,...,...,...,...,...,...,...,...,...
95,I predict by the end of the 2020’s and into the 2030’s war will be fought over the rarest and most sacred resource to life: Water.,"{\n ""0"": [""predict"", ""will"", ""be"", ""fought"", ""over"", ""the"", ""rarest"", ""and"", ""most"", ""sacred"", ""resource"", ""to"", ""life:""],\n ""1"": [""I""],\n ""2"": [""war""],\n ""3"": [""by the end of the 2020’s and into the 2030’s""],\n ""4"": [""Water""]\n}",openai/gpt-oss-120b,"predict, will, be, fought, over, the, r